model fine tuning using 433 dataset
testing model accuracy
testing model
zipping model
loading that zipped model to drive
unzipping the model
testing model with hugging face t5 and flan t5
testing model with hugging face t5 and flan t5

model fine tuning using 433 dataset

In [ ]:
!pip install transformers datasets sacremoses rouge-score
!pip install evaluate

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d3693441cb1a74061e14e53072f9a550c3a8c4ceb05fb5480890481ab7f7255d
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.9 MB/s eta 0:00:00


In [ ]:
!pip install transformers datasets sacremoses psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatib

In [ ]:
# %% [code]
# Uncomment this line to install required libraries if not already installed
# !pip install transformers datasets sacremoses psutil

import os
import time
import psutil
from datetime import datetime

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    TrainerCallback,
)
from datasets import load_dataset

# -------------------------------------------
# Helper function for timestamped logging
def print_time(message):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

# -------------------------------------------
# Custom callback for logging training loss (using on_log only)
class PrintCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print_time(f"[LOG] Global Step: {state.global_step} => Loss: {logs['loss']:.4f}")

# Callback to log memory usage and evaluation metrics (if available)
class MemoryUsageCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        mem = psutil.virtual_memory()
        print(f"[LOG] {time.ctime()} | Epoch: {state.epoch:.2f} | Step: {state.global_step} | "
              f"Memory used: {mem.used / (1024**3):.2f}GB / {mem.total / (1024**3):.2f}GB")
        if logs and "loss" in logs:
            print(f"[LOG] Step: {state.global_step} | Loss: {logs['loss']:.4f}")
        if logs and "eval_accuracy" in logs:
            print(f"[LOG] Step: {state.global_step} | Accuracy: {logs['eval_accuracy']:.4f}")

    def on_step_end(self, args, state, control, **kwargs):
        mem = psutil.virtual_memory()
        print(f"[STEP] After step {state.global_step}: Memory used: {mem.used / (1024**3):.2f}GB")

# Additional callback to print overall training progress with extra metrics if needed.
class TrainingProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        print(f"[PROGRESS] {time.ctime()} | Epoch: {state.epoch:.2f} | Global Step: {state.global_step}")
        if logs:
            for key, value in logs.items():
                print(f"    {key}: {value}")

# -------------------------------------------
# 1. Load the Quiz Dataset
print_time("Loading quiz dataset from finetune_dataset.jsonl...")
# Ensure your 'finetune_dataset.jsonl' file is in the current working directory.
dataset = load_dataset("json", data_files={"train": "finetune_dataset.jsonl"}, split="train")
print_time(f"Loaded {len(dataset)} training records.")

# -------------------------------------------
# 2. Load T5 Tokenizer and Model
print_time("Loading T5 tokenizer and model (t5-base)...")
model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)  # , use_auth_token="YOUR_TOKEN"  # Uncomment if needed
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  # , use_auth_token="YOUR_TOKEN"  # Uncomment if needed

# Enable gradient checkpointing to reduce memory usage during backpropagation.
if torch.cuda.is_available():
    model = model.to("cuda")
    model.gradient_checkpointing_enable()  # Trades compute for lower memory usage
    model.config.use_cache = False         # Disable cache to further reduce memory footprint
    print_time("Model moved to GPU with gradient checkpointing enabled and cache disabled.")

# -------------------------------------------
# 3. Preprocessing: Tokenize the Dataset
print_time("Tokenizing dataset...")
max_input_length = 512   # Adjust maximum input length if required
max_output_length = 128  # Adjust maximum output (target) length if required

def preprocess_function(examples):
    inputs = examples["prompt"]
    targets = examples["completion"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")

    # Tokenize target text; using text_target in recent versions.
    labels = tokenizer(text_target=targets, max_length=max_output_length, truncation=True, padding="max_length")
    # Replace padding token IDs with -100 so they are ignored during loss computation.
    labels["input_ids"] = [
        [label if label != tokenizer.pad_token_id else -100 for label in labels_example]
        for labels_example in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print_time("Mapping tokenization function over the dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
print_time("Tokenization complete.")

# -------------------------------------------
# 4. Set Up Training Arguments
print_time("Setting training arguments...")
# Adjusting batch sizes and accumulation steps to help keep memory usage under 9GB.
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-quiz-finetune",
    overwrite_output_dir=True,
    num_train_epochs=3,                         # Use more epochs for better performance if needed
    per_device_train_batch_size=2,              # Lower batch size to conserve memory
    gradient_accumulation_steps=2,              # Effective batch size = 2*2 = 4
    evaluation_strategy="no",                   # Change to "epoch" if you add evaluation data
    save_steps=500,
    save_total_limit=2,
    fp16=True,                                  # Mixed precision helps lower GPU memory usage
    logging_steps=10,
    report_to="none",
)
print_time("Training arguments set.")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# -------------------------------------------
# 5. Initialize the Seq2SeqTrainer with Custom Callbacks
print_time("Initializing the Seq2SeqTrainer...")
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[PrintCallback(), MemoryUsageCallback(), TrainingProgressCallback()],
    compute_metrics=None,  # Optionally, specify compute_metrics here if you have evaluation metrics
)

# -------------------------------------------
# 6. Start the Training Process
print_time("Starting training...")
start_time = time.time()
train_result = trainer.train()
end_time = time.time()
print_time(f"Training complete in {(end_time - start_time) / 60:.2f} minutes.")

# -------------------------------------------
# 7. Save the Fine-Tuned Model and Tokenizer
print_time("Saving fine-tuned model and tokenizer...")
trainer.save_model("./t5-quiz-finetune")
tokenizer.save_pretrained("./t5-quiz-finetune")
print_time("Fine-tuning process finished.")


[2025-04-14 06:49:46] Loading quiz dataset from finetune_dataset.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

[2025-04-14 06:49:46] Loaded 433 training records.
[2025-04-14 06:49:46] Loading T5 tokenizer and model (t5-base)...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

...

[2025-04-14 06:50:10] Tokenization complete.
[2025-04-14 06:50:10] Setting training arguments...
[2025-04-14 06:50:10] Training arguments set.
[2025-04-14 06:50:10] Initializing the Seq2SeqTrainer...
[2025-04-14 06:50:10] Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-4-6bfc71a3efab>:126: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


[STEP] After step 1: Memory used: 2.55GB


Step,Training Loss
10,2.717700
20,2.416200
30,2.028300
40,1.911600
50,1.994700
60,1.902000
70,1.663600
80,1.672400
90,1.639300
100,1.578900


[STEP] After step 2: Memory used: 2.57GB
[STEP] After step 3: Memory used: 2.58GB
[STEP] After step 4: Memory used: 2.61GB
[STEP] After step 5: Memory used: 2.61GB
[STEP] After step 6: Memory used: 2.60GB
[STEP] After step 7: Memory used: 2.60GB
[STEP] After step 8: Memory used: 2.61GB
[STEP] After step 9: Memory used: 2.63GB
[STEP] After step 10: Memory used: 2.61GB
[2025-04-14 06:50:22] [LOG] Global Step: 10 => Loss: 2.7177
[LOG] Mon Apr 14 06:50:22 2025 | Epoch: 0.09 | Step: 10 | Memory used: 2.61GB / 12.67GB
[LOG] Step: 10 | Loss: 2.7177
[PROGRESS] Mon Apr 14 06:50:22 2025 | Epoch: 0.09 | Global Step: 10
    loss: 2.7177
    grad_norm: 6.647473335266113
    learning_rate: 4.8611111111111115e-05
    epoch: 0.09216589861751152
[STEP] After step 11: Memory used: 2.62GB
[STEP] After step 12: Memory used: 2.62GB
[STEP] After step 13: Memory used: 2.61GB
[STEP] After step 14: Memory used: 2.61GB
[STEP] After step 15: Memory used: 2.61GB
[STEP] After step 16: Memory used: 2.61GB
[STEP] Af

testing model accuracy

In [ ]:
!pip install transformers datasets torch


In [ ]:
!pip install rouge-score


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Load tokenizer and model
model_path = "./t5-quiz-finetune"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load evaluation dataset
dataset = load_dataset("json", data_files={"eval": "finetune_dataset_validation.jsonl"}, split="eval")

# Setup
exact_matches, total = 0, 0
bleu_scores = []
rouge_scores = []
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method4

print("⏳ Evaluating...")

# Evaluation loop
for i, example in enumerate(dataset):
    prompt = example["prompt"]
    expected = example["completion"].strip().lower()

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=128)
    predicted = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    if predicted == expected:
        exact_matches += 1

    ref = expected.split()
    hyp = predicted.split()
    bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smoothie))

    try:
        rouge_scores.append(scorer.score(expected, predicted)["rougeL"].fmeasure)
    except:
        rouge_scores.append(0)

    total += 1
    if i % 10 == 0:
        print(f"Processed {i}/{len(dataset)} examples...")

# Final Results
accuracy = exact_matches / total * 100
avg_bleu = sum(bleu_scores) / total * 100
avg_rouge = sum(rouge_scores) / total * 100

print("\n✅ Evaluation Complete")
print(f"Accuracy (Exact Match): {accuracy:.2f}%")
print(f"BLEU Score (avg): {avg_bleu:.2f}")
print(f"ROUGE-L F1 Score (avg): {avg_rouge:.2f}")


Generating eval split: 0 examples [00:00, ? examples/s]

⏳ Evaluating...
Processed 0/87 examples...
Processed 10/87 examples...
Processed 20/87 examples...
Processed 30/87 examples...
Processed 40/87 examples...
Processed 50/87 examples...
Processed 60/87 examples...
Processed 70/87 examples...
Processed 80/87 examples...

✅ Evaluation Complete
Accuracy (Exact Match): 1.15%
BLEU Score (avg): 11.42
ROUGE-L F1 Score (avg): 33.45


testing model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from datasets import load_dataset

# Load tokenizer and model
model_path = "./t5-quiz-finetune"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load validation dataset
dataset = load_dataset("json", data_files={"eval": "finetune_dataset_validation.jsonl"}, split="eval")

print("\n--- SAMPLE OUTPUTS (Ground Truth vs Model Prediction) ---\n")

for idx, example in enumerate(dataset):
    if idx >= 5:
        break
    prompt = example["prompt"]
    expected = example["completion"].strip()

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=128)
    predicted = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    print(f"[{idx+1}] Prompt:\n{prompt}\n")
    print(f"✅ Expected:\n{expected}\n")
    print(f"🤖 Predicted:\n{predicted}\n")
    print("-" * 60)



--- SAMPLE OUTPUTS (Ground Truth vs Model Prediction) ---

[1] Prompt:
Generate quiz question: Corrected Version:

This text is unsuitable for quiz generation in its current form due to several issues:

1. **Typos and Formatting:**  The text contains numerous typos ("mer cury," "r eactive," inconsistent spacing, etc.) and formatting inconsistencies.  These errors would need to be corrected before questions could be reliably written.

2. **Inconsistent Numbering:** The section headings (3.4.3, 3.4.4) suggest a larger context that is missing.  This makes it difficult to create questions that accurately assess understanding within the broader topic.

3. **Lack of Clear Questions:** The text provides information but doesn't naturally lend itself to specific questions without significant restructuring.  For example, while it describes the process of roasting and calcination, it doesn't clearly state the *difference* between them in a way that would make a good quiz question.

4. **Chemical

zipping model

In [ ]:
import shutil

# Define the directory you want to zip and the output filename (without extension)
model_dir = "./t5-quiz-finetune"
output_filename = "t5-quiz-finetune"

# Create a ZIP archive of the model directory
shutil.make_archive(output_filename, 'zip', model_dir)

print(f"Model directory '{model_dir}' has been zipped to '{output_filename}.zip'.")


Model directory './t5-quiz-finetune' has been zipped to 't5-quiz-finetune.zip'.


loading that zipped model to drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import shutil for archiving
import shutil

# Set the local model directory and the output path on Drive
model_dir = "./t5-quiz-finetune"
# Change 'My Drive' to a subfolder if needed
output_drive_path = "/content/drive/My Drive/t5-quiz-finetune"

# Create a ZIP archive from the model directory directly into Drive
shutil.make_archive(output_drive_path, 'zip', model_dir)

print(f"Model directory '{model_dir}' has been zipped and saved as '{output_drive_path}.zip' in your Google Drive.")


Mounted at /content/drive
Model directory './t5-quiz-finetune' has been zipped and saved as '/content/drive/My Drive/t5-quiz-finetune.zip' in your Google Drive.


unzipping the model

In [ ]:
# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define paths
zip_path = '/content/drive/My Drive/t5-quiz-finetune.zip'  # Path to your zipped model in Drive
extract_path = '/content/t5-quiz-finetune'  # Local directory where the zip will be extracted

# Unzip the file if the extraction directory doesn't already exist
if not os.path.exists(extract_path):
    print("Unzipping the model from Drive...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print("Model directory already exists.")

# Load the model and tokenizer from the extracted directory
tokenizer = AutoTokenizer.from_pretrained(extract_path)
model = AutoModelForSeq2SeqLM.from_pretrained(extract_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded and ready for inference.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping the model from Drive...
Model loaded and ready for inference.


testing model with hugging face t5 and flan t5

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Revised prompt: Explicit instructions and fixed format
prompt = (
    "Generate one multiple-choice quiz question based on the following text. "
    "The output must strictly follow this format exactly, with four options labeled (A), (B), (C) and (D), "
    "and the correct answer indicated by its letter. Do not output any additional text.\n\n"
    "Format:\n"
    "Question: <The quiz question>\n"
    "Options:\n"
    "(A) <Option A text>\n"
    "(B) <Option B text>\n"
    "(C) <Option C text>\n"
    "(D) <Option D text>\n"
    "Correct Answer: <Correct option letter>\n\n"
    "Text: \"In this reaction you can observe that a single reactant breaks down to give simpler products. "
    "This is a decomposition reaction. Ferrous sulphate crystals (FeSO4·7H2O) lose water when heated and the colour of the crystals changes. "
    "It then decomposes to ferric oxide (Fe2O3), sulphur dioxide (SO2) and sulphur trioxide (SO3). "
    "Ferric oxide is a solid, while SO2 and SO3 are gases. Decomposition of calcium carbonate to calcium oxide and carbon dioxide on heating is "
    "an important decomposition reaction used in various industries. Calcium oxide is called lime or quick lime. It has many uses – one is in the manufacture of cement. "
    "When a decomposition reaction is carried out by heating, it is called thermal decomposition.\""
)

def generate_quiz(tokenizer, model, prompt, device, max_length=200, num_beams=3):
    inputs = tokenizer.encode(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(inputs, max_length=max_length, num_beams=num_beams, early_stopping=True)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# ---------------------------
# 1. Local Fine-Tuned Model
local_model_path = "./t5-quiz-finetune"
local_tokenizer = AutoTokenizer.from_pretrained(local_model_path)
local_model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path)
local_model.to(device)
local_model.eval()

# ---------------------------
# 2. Hugging Face T5-Base Model ("google-t5/t5-base")
hf_t5_tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
hf_t5_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base")
hf_t5_model.to(device)
hf_t5_model.eval()

# ---------------------------
# 3. Hugging Face FLAN-T5-Base Model ("google/flan-t5-base")
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
flan_model.to(device)
flan_model.eval()

# Generate outputs for each model and time them
start_time = time.time()
local_output = generate_quiz(local_tokenizer, local_model, prompt, device)
local_time = time.time() - start_time

start_time = time.time()
hf_t5_output = generate_quiz(hf_t5_tokenizer, hf_t5_model, prompt, device)
hf_t5_time = time.time() - start_time

start_time = time.time()
flan_output = generate_quiz(flan_tokenizer, flan_model, prompt, device)
flan_time = time.time() - start_time

# Print all outputs
print("---------- Local Fine-Tuned Model Output ----------")
print(local_output)
print(f"[Generation Time: {local_time:.2f} seconds]\n")

print("---------- Hugging Face T5-Base Output ----------")
print(hf_t5_output)
print(f"[Generation Time: {hf_t5_time:.2f} seconds]\n")

print("---------- Hugging Face FLAN-T5-Base Output ----------")
print(flan_output)
print(f"[Generation Time: {flan_time:.2f} seconds]\n")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

---------- Local Fine-Tuned Model Output ----------
Question: Which of the following is an important decomposition reaction used in various industries? Options: Ferrous sulphate crystals (FeSO47H2O) lose water when heated and the colour of the crystals changes. | Ferric oxide (Fe2O3) | Sulphur dioxide (SO2) Correct Answer: 2
[Generation Time: 15.75 seconds]

---------- Hugging Face T5-Base Output ----------
True
[Generation Time: 2.10 seconds]

---------- Hugging Face FLAN-T5-Base Output ----------
Which of the following is true according to the passage?Options:A Ferrous sulphate crystals lose water when heated.B Ferrous sulphate crystals lose water when heated.C Ferrous sulphate crystals lose water when heated.D Ferrous sulphate crystals lose water when heated.Answer:A
[Generation Time: 15.23 seconds]



testing model with hugging face t5 and flan t5

In [ ]:
pip install torch transformers datasets nltk rouge-score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 683.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
 

In [ ]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import nltk
from tqdm.notebook import tqdm  # Use tqdm for notebooks

# Download necessary nltk data (if not already done)
nltk.download('punkt')

def compute_token_f1(reference, prediction):
    """Compute a simple token-level F1 score based on word overlap."""
    ref_tokens = reference.split()
    pred_tokens = prediction.split()
    if not ref_tokens or not pred_tokens:
        return 0.0
    common = set(ref_tokens).intersection(set(pred_tokens))
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def generate_prediction(prompt_text, tokenizer, model, device, max_length=250, num_beams=3):
    inputs = tokenizer.encode(prompt_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(inputs, max_length=max_length, num_beams=num_beams, early_stopping=True)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
    return prediction

# ---------------------------
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Define the evaluation prompt text (you can use the prompt stored in your dataset, this is just an example)
# (For this evaluation loop we use the 'prompt' field from your dataset.)

# ---------------------------
# Load Evaluation Dataset
# We select only the first 20 examples for quick testing
eval_dataset = load_dataset("json", data_files={"eval": "finetune_dataset_validation.jsonl"}, split="eval").select(range(20))

# ---------------------------
# Load Models and Tokenizers

# 1. Local Fine-Tuned Model
local_model_path = "./t5-quiz-finetune"
local_tokenizer = AutoTokenizer.from_pretrained(local_model_path)
local_model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path)
local_model.to(device)
local_model.eval()

# 2. Hugging Face T5-Base Model ("google-t5/t5-base")
hf_t5_tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
hf_t5_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base")
hf_t5_model.to(device)
hf_t5_model.eval()

# 3. Hugging Face FLAN-T5-Base Model ("google/flan-t5-base")
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
flan_model.to(device)
flan_model.eval()

# ---------------------------
# Initialize metrics for each model
metrics = {
    "local": {"exact": 0, "bleu": [], "rouge": [], "meteor": [], "f1": []},
    "hf_t5": {"exact": 0, "bleu": [], "rouge": [], "meteor": [], "f1": []},
    "flan": {"exact": 0, "bleu": [], "rouge": [], "meteor": [], "f1": []},
}
total = 0
smoothie = SmoothingFunction().method4
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# ---------------------------
# Evaluation Loop (with progress bar)
for example in tqdm(eval_dataset, desc="Evaluating examples"):
    total += 1
    prompt_text = example["prompt"]
    reference = example["completion"].strip().lower()

    # Generate predictions for each model
    local_pred = generate_prediction(prompt_text, local_tokenizer, local_model, device)
    hf_t5_pred = generate_prediction(prompt_text, hf_t5_tokenizer, hf_t5_model, device)
    flan_pred = generate_prediction(prompt_text, flan_tokenizer, flan_model, device)

    # Exact match tracking
    if local_pred == reference:
        metrics["local"]["exact"] += 1
    if hf_t5_pred == reference:
        metrics["hf_t5"]["exact"] += 1
    if flan_pred == reference:
        metrics["flan"]["exact"] += 1

    # BLEU Scores
    ref_tokens = reference.split()
    metrics["local"]["bleu"].append(sentence_bleu([ref_tokens], local_pred.split(), smoothing_function=smoothie))
    metrics["hf_t5"]["bleu"].append(sentence_bleu([ref_tokens], hf_t5_pred.split(), smoothing_function=smoothie))
    metrics["flan"]["bleu"].append(sentence_bleu([ref_tokens], flan_pred.split(), smoothing_function=smoothie))

    # ROUGE-L F1 Scores
    try:
        metrics["local"]["rouge"].append(scorer.score(reference, local_pred)["rougeL"].fmeasure)
    except Exception:
        metrics["local"]["rouge"].append(0)
    try:
        metrics["hf_t5"]["rouge"].append(scorer.score(reference, hf_t5_pred)["rougeL"].fmeasure)
    except Exception:
        metrics["hf_t5"]["rouge"].append(0)
    try:
        metrics["flan"]["rouge"].append(scorer.score(reference, flan_pred)["rougeL"].fmeasure)
    except Exception:
        metrics["flan"]["rouge"].append(0)

    # METEOR Scores
    try:
        metrics["local"]["meteor"].append(meteor_score([reference], local_pred))
    except Exception:
        metrics["local"]["meteor"].append(0)
    try:
        metrics["hf_t5"]["meteor"].append(meteor_score([reference], hf_t5_pred))
    except Exception:
        metrics["hf_t5"]["meteor"].append(0)
    try:
        metrics["flan"]["meteor"].append(meteor_score([reference], flan_pred))
    except Exception:
        metrics["flan"]["meteor"].append(0)

    # Token-level F1 Scores
    metrics["local"]["f1"].append(compute_token_f1(reference, local_pred))
    metrics["hf_t5"]["f1"].append(compute_token_f1(reference, hf_t5_pred))
    metrics["flan"]["f1"].append(compute_token_f1(reference, flan_pred))

# ---------------------------
# Compute and Print Average Metrics for each model
def print_metrics(model_label, metric_dict, total_count):
    exact_pct = metric_dict["exact"] / total_count * 100 if total_count > 0 else 0
    avg_bleu = sum(metric_dict["bleu"]) / total_count * 100 if total_count > 0 else 0
    avg_rouge = sum(metric_dict["rouge"]) / total_count * 100 if total_count > 0 else 0
    avg_meteor = sum(metric_dict["meteor"]) / total_count * 100 if total_count > 0 else 0
    avg_f1 = sum(metric_dict["f1"]) / total_count * 100 if total_count > 0 else 0
    print(f"---------- {model_label.upper()} Model Evaluation ----------")
    print(f"Exact Match Accuracy: {exact_pct:.2f}%")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average ROUGE-L F1 Score: {avg_rouge:.2f}")
    print(f"Average METEOR Score: {avg_meteor:.2f}")
    print(f"Average Token-level F1 Score: {avg_f1:.2f}\n")

print_metrics("local", metrics["local"], total)
print_metrics("hf_t5", metrics["hf_t5"], total)
print_metrics("flan", metrics["flan"], total)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Evaluating examples:   0%|          | 0/20 [00:00<?, ?it/s]

---------- LOCAL Model Evaluation ----------
Exact Match Accuracy: 0.00%
Average BLEU Score: 13.00
Average ROUGE-L F1 Score: 36.96
Average METEOR Score: 0.00
Average Token-level F1 Score: 33.04

---------- HF_T5 Model Evaluation ----------
Exact Match Accuracy: 0.00%
Average BLEU Score: 0.00
Average ROUGE-L F1 Score: 0.18
Average METEOR Score: 0.00
Average Token-level F1 Score: 0.00

---------- FLAN Model Evaluation ----------
Exact Match Accuracy: 0.00%
Average BLEU Score: 0.56
Average ROUGE-L F1 Score: 12.12
Average METEOR Score: 0.00
Average Token-level F1 Score: 11.39

